In [ ]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
suppressMessages(library(dplyr))
suppressMessages(library(Seurat))
suppressMessages(library(ggplot2))
suppressMessages(library(sctransform))
suppressMessages(library(scater))
suppressMessages(library(reticulate))
suppressMessages(library(future))
suppressMessages(library('Biobase'))
suppressMessages(library(pheatmap))
suppressMessages(library(gplots))
suppressMessages(library('hdf5r'))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(EnsDb.Mmusculus.v79))
suppressMessages(library(BiocParallel))
suppressMessages(library(tictoc))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(ggplot2))
suppressMessages(library(cowplot))
suppressMessages(library(GenomeInfoDb))
suppressMessages(library("Signac"))

In [ ]:
samples <- c('MM_339','MM_340','MM_341','MM_342',
           'MM_343', 'MM_344', 'MM_345','MM_365',
           'MM_366','MM_367', 'MM_371', 'MM_383',
           'MM_384','MM_385','MM_386','MM_387',
           'MM_388','MM_390', 'MM_391','MM_392',
           'MM_394','MM_395','MM_396','MM_460',
           'MM_535','MM_536','MM_537','MM_543',
           'MM_544','MM_545','MM_546','MM_547',
           'MM_660','MM_661','MM_662','MM_664',
           'MM_665','MM_666','MM_667','6229_sort')

In [1]:
# file to file with the barcodes you want to KEEP! make sure they have sample ID on the front and '-1'
good_bcs <- file.path('/nfs/lab/rlmelton/npod/signac/2022jan/atac/31jan22/all_atac_oldSampleID_corrected.barcodes')
good <- trimws(scan(good_bcs, what='', sep='\n'), which='right', whitespace=' ')


In [ ]:
#### READ IN JOSHS MATRICES and then I have a ghetto work around because I'm dumb in R
#read in ATAC data from the lfm matrices (sm workaround method for now)
# load in starting ATAC long format matrices to a list(CellRanger filtered for now)
atacs_OG <- list()
atacs_FINAL <- list()

for (sample in samples) {
    #print(sample)
    wd <- sprintf('/nfs/lab/elisha/nPOD_output/atac/hg38/minread_1k/%s/', sample)
    atacs_OG[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    #atacs_OG[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V1)
    #atacs_OG[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    atacs_FINAL[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V1)
    atacs_OG[[sample]] <- NULL
    #atacs_FINAL[[sample]] <- atacs_FINAL[[sample]][atacs_FINAL[[sample]]$V2 %in% good,]
}

atac_mod <- atacs_FINAL

In [ ]:
#### ADD '-1' TO THE BARCODES FROM JOSHS MATRICES 
# THIS IS ALL BECAUSE THE FRAGMENT FILE FROM CELLRANGER IS IN THIS FORMAT AND SEURAT NEEDS THEM TO MATCH
atac_mod$MM_339$V2 <- paste0(atac_mod$MM_339$V2, "-1")
atac_mod$MM_340$V2 <- paste0(atac_mod$MM_340$V2, "-1")
atac_mod$MM_341$V2 <- paste0(atac_mod$MM_341$V2, "-1")
atac_mod$MM_342$V2 <- paste0(atac_mod$MM_342$V2, "-1")
atac_mod$MM_343$V2 <- paste0(atac_mod$MM_343$V2, "-1")
atac_mod$MM_344$V2 <- paste0(atac_mod$MM_344$V2, "-1")
atac_mod$MM_345$V2 <- paste0(atac_mod$MM_345$V2, "-1")

atac_mod$MM_365$V2 <- paste0(atac_mod$MM_365$V2, "-1")
atac_mod$MM_366$V2 <- paste0(atac_mod$MM_366$V2, "-1")
atac_mod$MM_367$V2 <- paste0(atac_mod$MM_367$V2, "-1")

atac_mod$MM_371$V2 <- paste0(atac_mod$MM_371$V2, "-1")
atac_mod$MM_383$V2 <- paste0(atac_mod$MM_383$V2, "-1")
atac_mod$MM_384$V2 <- paste0(atac_mod$MM_384$V2, "-1")
atac_mod$MM_385$V2 <- paste0(atac_mod$MM_385$V2, "-1")
atac_mod$MM_386$V2 <- paste0(atac_mod$MM_386$V2, "-1")
atac_mod$MM_387$V2 <- paste0(atac_mod$MM_387$V2, "-1")
atac_mod$MM_388$V2 <- paste0(atac_mod$MM_388$V2, "-1")

atac_mod$MM_390$V2 <- paste0(atac_mod$MM_390$V2, "-1")
atac_mod$MM_391$V2 <- paste0(atac_mod$MM_391$V2, "-1")
atac_mod$MM_392$V2 <- paste0(atac_mod$MM_392$V2, "-1")
atac_mod$MM_394$V2 <- paste0(atac_mod$MM_394$V2, "-1")
atac_mod$MM_395$V2 <- paste0(atac_mod$MM_395$V2, "-1")
atac_mod$MM_396$V2 <- paste0(atac_mod$MM_396$V2, "-1")
atac_mod$MM_460$V2 <- paste0(atac_mod$MM_460$V2, "-1")

atac_mod$MM_535$V2 <- paste0(atac_mod$MM_535$V2, "-1")
atac_mod$MM_536$V2 <- paste0(atac_mod$MM_536$V2, "-1")
atac_mod$MM_537$V2 <- paste0(atac_mod$MM_537$V2, "-1")
atac_mod$MM_543$V2 <- paste0(atac_mod$MM_543$V2, "-1")
atac_mod$MM_544$V2 <- paste0(atac_mod$MM_544$V2, "-1")
atac_mod$MM_545$V2 <- paste0(atac_mod$MM_545$V2, "-1")
atac_mod$MM_546$V2 <- paste0(atac_mod$MM_546$V2, "-1")
atac_mod$MM_547$V2 <- paste0(atac_mod$MM_547$V2, "-1")

atac_mod$MM_660$V2 <- paste0(atac_mod$MM_660$V2, "-1")
atac_mod$MM_661$V2 <- paste0(atac_mod$MM_661$V2, "-1")
atac_mod$MM_662$V2 <- paste0(atac_mod$MM_662$V2, "-1")
atac_mod$MM_664$V2 <- paste0(atac_mod$MM_664$V2, "-1")
atac_mod$MM_665$V2 <- paste0(atac_mod$MM_665$V2, "-1")
atac_mod$MM_666$V2 <- paste0(atac_mod$MM_666$V2, "-1")
atac_mod$MM_667$V2 <- paste0(atac_mod$MM_667$V2, "-1")
atac_mod$`6229_sort`$V2 <- paste0(atac_mod$`6229_sort`$V2, "-1")


In [ ]:
### GRAB THE WINDOWS FILE FROM JOSH'S PIPELINE 
#read in the HVWs set, we'll cut down all samples to be these windows
hvw_fp2 <- '/nfs/lab/elisha/nPOD_output/atac/hg38/clustering/merge_sept30/adata_merged_hvg_24jan22.txt'
hvws2 <- scan(hvw_fp2, what="", sep="\n")
print(head(hvws2))

#sort alphanumerically
hvws_fin2 <- sort(hvws2)
print(head(hvws_fin2))

In [ ]:
#function which takes in a list of long format atac_fragment dfs with 
#sample names (df), an overall windows file (windows) and then makes these 
#into sparse matrices and merges them together

#modified to take in the hvws set and use those... will still check if 
#there's any missing windows and add those, so only a few changes!

merge_sparse_matrices_hvws <- function(dfs, windows){
    samples <- names(dfs)
    for (sample in samples){
        #get missing windows list for this sample
        print(paste(sample,Sys.time(),sep=': '))
        df <- dfs[[sample]]
        mis_windows <- windows[!windows %in% levels(df$V1)]
        
        #make sure there are missing_windows
        if (length(mis_windows) > 0){
            print('Adding missing windows')
            #create a new long format matrix (sm) with the missing windows added as 0 counts
            filler_bc <- as.character(df$V2[[1]])
            print(paste("Using filler BC:",filler_bc,sep=" "))
            new_rows <- cbind(as.data.frame(mis_windows),
                              as.data.frame(rep(filler_bc),length(mis_windows)),
                              as.data.frame(rep(0,length(mis_windows))))
            colnames(new_rows) <- c("V1","V2","V3")
            lfm <- rbind(df,new_rows)
        #if there aren't, set lfm to df
        } else {
            print('No windows were missing')
            lfm <- df
        }
        
        #cut down lfm to just be the hvws (windows)
        lfm_cut <- lfm[lfm$V1 %in% windows,]
        
        #cut down barcodes to keep those in final set 
        #lfm_cut <- lfm_cut[lfm_cut$V2 %in% good,]
        #lfm_cut <- lfm[,lfm$V2 %in% good]
        #atacs_FINAL[[sample]] <- atacs_FINAL[[sample]][atacs_FINAL[[sample]]$V2 %in% good,]
        #set the levels of the lfm based on the desired bc order and reorder V1
        lfm_cut$V1 <- factor(lfm_cut$V1, levels=windows)
        lfm2 <- lfm_cut[order(lfm_cut$V1),]
        lfm2$V2 <- as.factor(lfm2$V2)
        
        if (sample == samples[1]){
            #if first sample, will make the overall sparse matrix 
            overall_sm <- with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(overall_sm))
            
        } else {
            #lfm2 <- lfm2[lfm2$V2 %in% good,]

            #otherwise, convert into a sparse matrix and add to the overall one
            sm = with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(sm))
            overall_sm = cbind(overall_sm, sm) 
        }
    }
    return(overall_sm)
}

In [ ]:
overall_sm2 <- merge_sparse_matrices_hvws(atac_mod,hvws_fin2)
#check if the windows of overall_sm are sorted (they should be)
sorted_windows2 <- sort(row.names(overall_sm2))
table(sorted_windows2 == row.names(overall_sm2))

#subset to good barcodes
overall_sm3 <- overall_sm2[,colnames(overall_sm2) %in% good]


In [ ]:
#continue to make Seurat compatible object
suppressMessages(annotations <- GetGRangesFromEnsDb(ensdb=EnsDb.Hsapiens.v86))
seqlevelsStyle(annotations) <- 'UCSC'
genome(annotations) <- 'hg38'

Create the merged fragment file using <br>
<code>for SAMPLE in scATAC_islet7453_tgn_1000 scATAC_islet4807_tgn_10000 scATAC_islet4807_unt_4000 scATAC_islet7453_unt_500 1000tgn 1000untreated; do zcat /nfs/lab/rlmelton/npod/integration/nov18_seuratIntegration/${SAMPLE}_fragments.tsv.gz | awk -v SAMPLE=$SAMPLE \ 'BEGIN{FS=OFS="\t"} {print $1,$2,$3,SAMPLE"_"$4,$5}\'; done | sort -k1,1 -k2,2n -S 64G | bgzip -c -@ 16 > /nfs/lab/rlmelton/npod/integration/nov18_seuratIntegration/merged.atac_fragments_new.tsv.gz 


tabix -p bed /nfs/lab/rlmelton/npod/integration/nov18_seuratIntegration/merged.atac_fragments_new.tsv.gz </code>
<br> pass the path to the fragment.tsv from cellranger <br>

In [ ]:
frag.file = '/nfs/lab/rlmelton/npod/integration/nov18_seuratIntegration/nPOD_nov29/fragment_files/merged.atac_fragments_test.tsv.gz'

In [ ]:
atac_assay <- CreateChromatinAssay(counts=overall_sm3, sep=c(':', '-'), genome='hg38', fragments=frag.file, min.cells=0, min.features=-1, annotation=annotations)
Atac <-CreateSeuratObject(atac_assay, project = "nPOD", assay = "ATAC",
  min.cells = 0, min.features = 0, names.field = 1,
  names.delim = "_")
Atac$library <- gsub('.{19}$', '', rownames(Atac@meta.data))                   

In [ ]:
# add metadata
sex_F = list('MM_665','MM_664','MM_661','MM_662','6229_sort',
'MM_385',
'MM_546',
'MM_547',
'MM_396',
'MM_367',
'MM_392',
'MM_395',
'MM_386',
'MM_394',
'MM_548',
'MM_343',
'MM_391',
'MM_365',
'MM_370',
'MM_383',
'MM_384',
'MM_387',
'MM_388')
cond_t1d = list('MM_665','MM_666','MM_661','MM_662',
'MM_339',
'MM_340',
'MM_343',
'MM_391',
'MM_365',
'MM_460',
'MM_370',
'MM_383',
'MM_384',
'MM_387',
'MM_388',
'MM_537')
cond_aab = list('MM_667','MM_342',
'MM_371',
'MM_385',
'MM_390',
'MM_535',
'MM_543',
'MM_544',
'MM_546',
'MM_547')
cond_ctrl = list('MM_660','MM_664','6229_sort','MM_396',
'MM_345',
'MM_367',
'MM_392',
'MM_395',
'MM_386',
'MM_341',
'MM_344',
'MM_366',
'MM_394',
'MM_536',
'MM_545',
'MM_548')

multiome = list('MM_665','MM_664','MM_661','MM_662','6229_sort','MM_660','MM_666','MM_667')

Atac@meta.data$technology[Atac@meta.data$library %in% multiome] <- 'multiome'
Atac@meta.data$technology[!Atac@meta.data$library %in% multiome] <- 'snatac'

Atac@meta.data$sex[Atac@meta.data$library %in% sex_F] <- 'F'
Atac@meta.data$sex[!Atac@meta.data$library %in% sex_F] <- 'M'

Atac@meta.data$condition[Atac@meta.data$library %in% cond_t1d] <- 'T1D'
Atac@meta.data$condition[Atac@meta.data$library %in% cond_aab] <- 'Aab'
Atac@meta.data$condition[Atac@meta.data$library %in% cond_ctrl] <- 'Control'


In [ ]:
### remove amulet barcodes
snatac_amulet <- file.path('/nfs/lab/rlmelton/npod/amulet/atac/final_barcodes/amulet_atac_removeBCs.txt')
multiome_amulet <- file.path('/nfs/lab/rlmelton/npod/amulet/multiome/final_removal_bcList/amulet_multiome_removeBCs.txt')


snatac_amulet1 <- trimws(scan(snatac_amulet, what='', sep='\n'), which='right', whitespace=' ')
multiome_amulet1 <- trimws(scan(multiome_amulet, what='', sep='\n'), which='right', whitespace=' ')

all.cells = c(snatac_amulet1,multiome_amulet1)

print(length(all.cells))
Atac$remove_cells <- (Cells(Atac) %in% all.cells)
Atac <- subset(Atac, subset=remove_cells==FALSE)
Atac

In [ ]:
saveRDS(Atac, file='/nfs/lab/rlmelton/npod/signac/2022jan/atac/31jan22/1feb22_nPOD_atac_v2_correctBCs.rds')


### Process data

In [ ]:
DefaultAssay(Atac) <- 'ATAC'
Atac <- RunTFIDF(Atac)
Atac <- FindTopFeatures(Atac, min.cutoff='q0', verbose=FALSE)
Atac <- RunSVD(Atac)

hm_atac <- HarmonyMatrix(Embeddings(Atac, reduction='lsi'),Atac@meta.data,  c("library","sex","technology"), do_pca=FALSE,plot_convergence = TRUE, verbose = TRUE)
Atac[['harmony.atac']] <- CreateDimReducObject(embeddings=hm_atac, key='atac_', assay='ATAC')
Atac <- RunUMAP(Atac, dims=2:30, reduction='harmony.atac', reduction.name='umap.atac', reduction.key='atacUMAP_')

Atac <- FindNeighbors(object = Atac, reduction = 'harmony.atac', dims = 2:30)
Atac <- FindClusters(object = Atac, algorithm=4,resolution = 0.5,method = "igraph") 

DimPlot(object = Atac, label = TRUE) + NoLegend()


In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
p1 <- VlnPlot(Atac, features='nCount_ATAC',  pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(Atac$nCount_ATAC), linetype='dashed')#
p2 <- VlnPlot(Atac, features='nFeature_ATAC', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(Atac$nFeature_ATAC), linetype='dashed')
p1 / p2 

options(repr.plot.width=10, repr.plot.height=10)
DimPlot(object = Atac, group.by = 'technology',label = FALSE)# + NoLegend()
DimPlot(object = Atac, group.by = 'library',label = FALSE)# + NoLegend()
DimPlot(object = Atac, group.by = 'condition',label = FALSE)# + NoLegend()



### Remove low quality clusters

In [ ]:
outdir2 = '/nfs/lab/rlmelton/npod/signac/2022jan/atac/31jan22/remove_harmony_30PCs/'
#write to file the BCs from clusters 18 and 19 (low quality)
lowqual_clusters = c(15)
to_remove <- file.path(outdir2,"harmony.lowqualclusters5.txt")
write.table(adata_new[[]][adata_new$seurat_clusters %in% lowqual_clusters, c()], 
            to_remove, col.names=FALSE, quote=FALSE)

In [ ]:
#read back in the low quality clusters and doublet BCs
to_remove <- file.path(outdir2,"harmony.lowqualclusters1.txt")


remove.cells1 <- trimws(scan(to_remove, what='', sep='\n'), which='right', whitespace=' ')

print(length(remove.cells1))


#remove low quality clusters
all.cells = c(remove.cells1)

print(length(all.cells))
Atac$remove_cells <- (Cells(Atac) %in% all.cells)
adata_new <- subset(Atac, subset=remove_cells==FALSE)
adata_new

### Recluster 

In [ ]:
DefaultAssay(adata_new) <- 'ATAC'
adata_new <- RunTFIDF(adata_new)
adata_new <- FindTopFeatures(adata_new, min.cutoff='q0', verbose=FALSE)
adata_new <- RunSVD(adata_new)

hm_atac <- HarmonyMatrix(Embeddings(adata_new, reduction='lsi'),adata_new@meta.data,  c("library","sex","technology"), do_pca=FALSE,plot_convergence = TRUE, verbose = TRUE)
adata_new[['harmony.atac']] <- CreateDimReducObject(embeddings=hm_atac, key='atac_', assay='ATAC')
adata_new <- RunUMAP(adata_new, dims=2:30, reduction='harmony.atac', reduction.name='umap.atac', reduction.key='atacUMAP_')

adata_new <- FindNeighbors(object = adata_new, reduction = 'harmony.atac', dims = 2:30)
adata_new <- FindClusters(object = adata_new, algorithm=4,resolution = 0.5,method = "igraph") 

DimPlot(object = adata_new, label = TRUE) + NoLegend()


In [ ]:
saveRDS()